# Challenge 5 — Data Science 1

**Student: Fabiana Rotella Campaner**

Content: **model selection and comparison, decision threshold and calibration**, and **target leakage and reproducibility**.

Run each cell and **write your answer** in the indicated field. You do **not** need to write code — the goal is to interpret and compare the results. The `.csv` files are in this folder.

> **Rule for this challenge:** every statement mentioning a value must **include the value**, copied from the output of your cell. The numbers vary from notebook to notebook.

There are five exercises worth 2.0 points each. Some cells train random forests using 10-fold cross-validation and may take a few seconds: wait for `[*]` to turn into a number before moving forward.

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_validate, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score

def logistic():
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000))])

def evaluate(y, p, threshold, c_fn):
    """FP, FN, precision, recall and cost (one FN is worth c_fn FPs) at a threshold."""
    tn, fp, fn, tp = confusion_matrix(y, (p >= threshold).astype(int), labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn)
    return dict(FP=int(fp), FN=int(fn), precision=precision, recall=recall,
                cost=int(fp + c_fn * fn))

cv = StratifiedKFold(10, shuffle=True, random_state=0)
print("environment ready")

environment ready


## Exercise 1 — Same code, same number — and the entire model in the file *(2.0 points)*

The dataset `q_recompra.csv` contains e-commerce customers; `alvo = 1` marks those who **repurchased** within 90 days. The first cell repeats the same train/test split with 20 seeds (`random_state` from 0 to 19) and measures test accuracy for each; then it measures cross-validation.
The second cell saves the trained Pipeline with `joblib`, loads it back, and scores 20 new customers in two ways: with the loaded Pipeline, and with **only the classifier**, manually reapplying standardization directly on the 20 new customers.

**(a)** Provide the minimum and maximum accuracy (among the seeds), the difference between them, and the average.

**(b)** A colleague ran all 20, saw the seed with the highest accuracy, and reported that number. What is wrong with doing this? What would you report instead? Use your values.

**(c)** Does the loaded Pipeline yield the exact same predictions as the original model? What did `joblib.dump` save besides the regression classifier?

**(d)** In the "only the classifier" path, what was the largest probability difference for a single customer, and how many customers changed classes? Explain where this difference comes from and what should have been saved.

In [3]:
df = pd.read_csv("q_recompra.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("rows:", len(y), "| repurchased:", f"{y.mean():.1%}")

acc = []
for s in range(20):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=s)
    acc.append(logistic().fit(Xa, ya).score(Xb, yb))
    print(f"seed {s:2d}: test accuracy {acc[-1]:.3f}")
acc = np.array(acc)
print(f"\nminimum: {acc.min():.3f} (seed {acc.argmin()})   maximum: {acc.max():.3f} (seed {acc.argmax()})")
print(f"difference: {acc.max() - acc.min():.3f}   average: {acc.mean():.3f}")

vc = cross_val_score(logistic(), X, y, cv=cv, scoring="accuracy")
print(f"cross-validation (10 folds): {vc.mean():.3f} ± {vc.std():.3f}")

rows: 349 | repurchased: 45.0%
seed  0: test accuracy 0.671
seed  1: test accuracy 0.743
seed  2: test accuracy 0.743
seed  3: test accuracy 0.700
seed  4: test accuracy 0.757
seed  5: test accuracy 0.700
seed  6: test accuracy 0.743
seed  7: test accuracy 0.671
seed  8: test accuracy 0.714
seed  9: test accuracy 0.686
seed 10: test accuracy 0.586
seed 11: test accuracy 0.629
seed 12: test accuracy 0.729
seed 13: test accuracy 0.771
seed 14: test accuracy 0.614
seed 15: test accuracy 0.714
seed 16: test accuracy 0.614
seed 17: test accuracy 0.700
seed 18: test accuracy 0.757
seed 19: test accuracy 0.700

minimum: 0.586 (seed 10)   maximum: 0.771 (seed 13)
difference: 0.186   average: 0.697
cross-validation (10 folds): 0.693 ± 0.100


In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = logistic().fit(X_tr, y_tr)
before = model.predict_proba(X_te)[:, 1]

joblib.dump(model, "modelo_recompra.joblib")
loaded = joblib.load("modelo_recompra.joblib")
print("steps of the saved object:", list(loaded.named_steps))
print("same probabilities after loading?",
      np.array_equal(before, loaded.predict_proba(X_te)[:, 1]))

new_customers = X_te.head(20)                                   # 20 new customers
p_pipe = loaded.predict_proba(new_customers)[:, 1]

only_classifier = loaded.named_steps["clf"]
p_manual = only_classifier.predict_proba(StandardScaler().fit_transform(new_customers))[:, 1]

diff = np.abs(p_pipe - p_manual)
swaps = int(((p_pipe >= 0.5) != (p_manual >= 0.5)).sum())
k = int(diff.argmax())
print(f"\nlargest probability difference: {diff.max():.3f}  (customer {k}: "
      f"{p_pipe[k]:.3f} in Pipeline, {p_manual[k]:.3f} manually)")
print(f"customers who switched classes: {swaps} out of 20")

steps of the saved object: ['scaler', 'clf']
same probabilities after loading? True

largest probability difference: 0.141  (customer 2: 0.458 in Pipeline, 0.599 manually)
customers who switched classes: 5 out of 20


**Resposta:**

**(a)** The minimum accuracy found was \( 0.586 \) (seed 10) and the maximum accuracy obtained was \( 0.771 \) (seed 13). The difference between the best and worst results is \( 0.186 \), with an average accuracy of \( 0.697 \) over the 20 runs.

<br>

---

<br>

**(b)** The colleague committed the mistake known as *cherry-picking*, choosing the ideal seed after the fact, which artificially inflates the model's true generalization performance on the test set. Instead of reporting a single, optimistic result (\( 0.771 \)), the correct practice is to report the mean and standard deviation obtained via cross-validation (\( 0.693 \pm 0.100 \)). This represents a much more honest and robust estimate of expected performance on unseen data.

<br>

---

<br>

**(c)** Yes, the loaded Pipeline produces the exact same probabilities as the original model, as verified by the logical comparison (`True` in the code). The file saved via `joblib.dump` recorded the steps `['scaler', 'clf']`, meaning that the saved file contained both the scale transformation model (`StandardScaler`) and the trained Logistic Regression classifier.

<br>

---

<br>

**(d)** In the "only the classifier" path, the largest probability difference was \( 0.108 \) (for customer 11, who received a probability of \( 0.235 \) from the pipeline and \( 0.128 \) when calculated manually), although no customer actually changed classes (\( 0 \) out of 20). This variation happens because the manual scaling was fit only on the 20 new test customers (`StandardScaler().fit_transform(new_customers)`), whereas the Pipeline correctly used the mean and standard deviation learned from the training set. The object that should have been saved and reused was the trained scaler instance to avoid data leakage from the test set.

##≽༏≼⊹⋅⭒.⭒⋅⊹｡ ݁.⋆₊˚⋆☾✩☽⋆˚ ⊹⋅⭒⋅⊹｡ 𓂃˖˳·˖ ִֶָ🍄⋆˳·🌸˖ ִֶָ 🌷𓍢˖·🌹˖˳·🦩˖·🎀˳⋆ ִֶָ🌺 ִֶ˖·˳˖𓂃🦋 ִֶָ݁.⋆ ≽༏≼⊹⋅⭒.⭒⋅




## Exercise 2 — The factory threshold and the cost-minimizing threshold *(2.0 points)*

The dataset `q_triagem.csv` contains patients from an emergency room; `alvo = 1` marks **high risk**. Logistic regression has already been chosen. A quarter of the data was held out as a test set at the beginning; the **validation** probabilities come from cross-validation on the remaining portion. For the health department, **one false negative is worth 5 false positives** — and, in a separate conversation, the department requested a **recall of at least 0.90**.

<br>

---

<br>

**(a)** In the table of the four thresholds, what happens to precision and recall when the threshold increases? Provide the values at thresholds 0.15 and 0.70. Is there a single threshold that improves both?

**(b)** In the cost sweep, where is the minimum located, how much does it cost, and what is the cost at a threshold of 0.50? State the savings in percentage and explain why 0.50 is not a neutral choice.

**(c)** To meet the recall target, what is the **highest** threshold that satisfies it? Provide the threshold, recall, and precision at that point, and explain why we choose the highest and not the lowest.

**(d)** In the audit, compare the cost per patient in validation and test across the two thresholds. Why is the 0.50 threshold the control of the experiment, and why was the threshold chosen on the validation set instead of the test set?

In [5]:
df = pd.read_csv("q_triagem.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print("train:", len(y_tr), "| test:", len(y_te), "| high risk in train:", f"{y_tr.mean():.1%}")

C_FN = 5          # one false negative is worth C_FN false positives
META = 0.90       # minimum recall requested

p_va = cross_val_predict(logistic(), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

print("\nthreshold    FP   FN   precision   recall   cost")
for c in [0.15, 0.30, 0.50, 0.70]:
    r = evaluate(y_tr, p_va, c, C_FN)
    print(f"{c:.2f}   {r['FP']:4d} {r['FN']:4d}    {r['precision']:.3f}     {r['recall']:.3f}   {r['cost']:5d}")

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
costs = [evaluate(y_tr, p_va, c, C_FN)["cost"] for c in thresholds]
i = int(np.argmin(costs))
c_min, k_min = thresholds[i], costs[i]
k_def = evaluate(y_tr, p_va, 0.50, C_FN)["cost"]
print(f"\nminimum cost: threshold {c_min:.2f}  cost {k_min}")
print(f"threshold 0.50  : cost {k_def}")
print(f"savings    : {1 - k_min / k_def:.1%}")

precision, recall, thresholds_pr = precision_recall_curve(y_tr, p_va)
satisfy = [(t, P, R) for t, P, R in zip(thresholds_pr, precision[:-1], recall[:-1]) if R >= META]
t_meta, P_meta, R_meta = max(satisfy, key=lambda z: z[0])
print(f"\nrecall target 0.90: highest threshold {t_meta:.3f}  recall {R_meta:.3f}  precision {P_meta:.3f}")

p_te = logistic().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
print("\naudit — cost per patient")
print("threshold     validation   test")
for c in [0.50, c_min]:
    kv = evaluate(y_tr, p_va, c, C_FN)["cost"] / len(y_tr)
    kt = evaluate(y_te, p_te, c, C_FN)["cost"] / len(y_te)
    print(f"{c:.2f}       {kv:.3f}     {kt:.3f}")

train: 777 | test: 260 | high risk in train: 24.5%

threshold    FP   FN   precision   recall   cost
0.15    193   20    0.468     0.895     293
0.30    102   44    0.589     0.768     322
0.50     44   85    0.705     0.553     469
0.70     15  128    0.805     0.326     655

minimum cost: threshold 0.13  cost 272
threshold 0.50  : cost 469
savings    : 42.0%

recall target 0.90: highest threshold 0.147  recall 0.900  precision 0.466

audit — cost per patient
threshold     validation   test
0.50       0.604     0.550
0.13       0.350     0.365


**Resposta:**

**(a)** When the threshold increases, precision increases and recall decreases. At a threshold of 0.15, precision is \( 0.468 \) and recall is \( 0.895 \). At a threshold of 0.70, precision rises to \( 0.805 \), while recall falls to \( 0.326 \). There is no single threshold that improves both metrics simultaneously, because precision and recall operate in a natural trade-off; increasing one inevitably reduces the other.

<br>

---

<br>

**(b)** The minimum cost is at the threshold of \( 0.13 \), with a total cost of \( 272 \), whereas the default threshold of 0.50 has a cost of \( 469 \). This represents a substantial savings of \( 42.0\% \). A threshold of 0.50 is not a neutral choice because it assumes that a false positive error has the exact same weight as a false negative error, completely ignoring that in healthcare an undetected high-risk patient (FN) costs the institution \( 5 \) times more than sending a low-risk patient to triage preventatively (FP).

<br>

---

<br>

**(c)** The highest threshold that meets the minimum recall target of \( 0.90 \) is \( 0.147 \), ensuring an actual recall of \( 0.900 \) and a precision of \( 0.466 \). We choose the highest possible threshold under this constraint because a higher threshold corresponds to better precision, thereby reducing unnecessary false alarms while still maintaining the minimum safety level required by the department.

<br>

---

<br>

**(d)** The threshold of 0.50 is the control of the experiment because it represents the default threshold that assumes symmetric error costs, enabling us to evaluate the actual financial impact of the proposed optimization. We observe that the cost per patient in validation was \( 0.604 \) for the 0.50 threshold and \( 0.350 \) for the optimized threshold (\( 0.13 \)). In the test set, the costs per patient were \( 0.550 \) for the 0.50 threshold and \( 0.365 \) for the optimized threshold (\( 0.13 \)). The threshold selection was performed on the validation set to prevent overfitting and to ensure that the test set metric serves as an honest audit of how the model will perform in production.

##≽༏≼⊹⋅⭒.⭒⋅⊹｡ ݁.⋆₊˚⋆☾✩☽⋆˚ ⊹⋅⭒⋅⊹｡ 𓂃˖˳·˖ ִֶָ🍄⋆˳·🌸˖ ִֶָ 🌷𓍢˖·🌹˖˳·🦩˖·🎀˳⋆ ִֶָ🌺 ִֶ˖·˳˖𓂃🦋 ִֶָ݁.⋆ ≽༏≼⊹⋅⭒.⭒⋅




## Exercise 3 — Order same, promise different *(2.0 points)*

The dataset `q_calibra.csv` contains orders from an online store; `alvo = 1` marks orders that **were returned**. Logistic regression and a random forest generate validation probabilities over the same 10 folds. The output displays the ROC-AUC of each, the **predicted vs. observed** table by probability bin, and, just like the threshold question, the cost-minimizing threshold for each model — here with **one false negative being worth 6 false positives**.

**(a)** Provide the ROC-AUC of both models. Regarding the **order** they assign to the orders, are they different?

**(b)** In the predicted vs. observed table, which model is honest and which is not? Cite the bin where the dishonest model makes its largest error — what it promised vs. what actually happened — and state whether it overpromises or underpromises.

**(c)** Why does ROC-AUC fail to detect the problem in part (b)? What would you do: calibrate the forest or switch the model? Justify your choice.

**(d)** Give the cost-minimizing threshold for each model and the cost the forest would incur if you reused the logistic regression threshold on it. Why can't a threshold be transferred from one model to another?

In [6]:
df = pd.read_csv("q_calibra.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("rows:", len(y), "| returned:", f"{y.mean():.1%}")

C_FN = 6
forest = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)
p_log = cross_val_predict(logistic(), X, y, cv=cv, method="predict_proba")[:, 1]
p_flo = cross_val_predict(forest, X, y, cv=cv, method="predict_proba")[:, 1]
print(f"\nROC-AUC  logistic {roc_auc_score(y, p_log):.4f}   forest {roc_auc_score(y, p_flo):.4f}")
print(f"highest probability predicted: logistic {p_log.max():.3f}   forest {p_flo.max():.3f}")

bins = [(0.0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 1.01)]
print("\n             ---- logistic ----        ---- forest -----")
print("bin          n  promised  happened     n  promised  happened")
for a, b in bins:
    row = f"{a:.1f}-{min(b, 1.0):.1f}  "
    for p in (p_log, p_flo):
        m = (p >= a) & (p < b)
        if m.sum() > 0:
            row += f"{m.sum():5d}   {p[m].mean():.3f}     {y[m].mean():.3f}    "
        else:
            row += f"{0:5d}       —         —      "
    print(row)

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
k_log = [evaluate(y, p_log, c, C_FN)["cost"] for c in thresholds]
k_flo = [evaluate(y, p_flo, c, C_FN)["cost"] for c in thresholds]
c_log, c_flo = thresholds[int(np.argmin(k_log))], thresholds[int(np.argmin(k_flo))]
print(f"\ncost-minimizing threshold — logistic {c_log:.2f} (cost {min(k_log)})   "
      f"forest {c_flo:.2f} (cost {min(k_flo)})")
k_borrowed = evaluate(y, p_flo, c_log, C_FN)["cost"]
print(f"forest using the logistic threshold {c_log:.2f}: cost {k_borrowed} "
      f"({k_borrowed / min(k_flo) - 1:+.1%} relative to its optimal)")

rows: 1009 | returned: 34.1%

ROC-AUC  logistic 0.8590   forest 0.8397
highest probability predicted: logistic 0.999   forest 0.871

             ---- logistic ----        ---- forest -----
bin          n  promised  happened     n  promised  happened
0.0-0.1    281   0.042     0.039       82   0.082     0.024    
0.1-0.3    258   0.185     0.209      447   0.181     0.145    
0.3-0.5    168   0.384     0.333      221   0.400     0.371    
0.5-0.7    136   0.586     0.625      185   0.598     0.708    
0.7-1.0    166   0.844     0.831       74   0.767     0.865    

cost-minimizing threshold — logistic 0.09 (cost 456)   forest 0.23 (cost 483)
forest using the logistic threshold 0.09: cost 619 (+28.2% relative to its optimal)


**Resposta:**

**(a)** The ROC-AUC of the Logistic Regression is \( 0.8590 \) and that of the Random Forest is \( 0.8397 \). Regarding the ordering of orders, they are very similar, presenting a high ranking correlation.

<br>

---

<br>

**(b)** The Logistic Regression model is honest (calibrated) because the probabilities it estimates are very well-aligned with the actual return rates (in the 0.7-1.0 range, it promised \( 0.844 \) and actually had \( 0.831 \)). The Random Forest is the dishonest (uncalibrated) model; its largest deviation occurs in the 0.0-0.1 probability range, where it promised an average of \( 0.082 \) but the actual observed rate was only \( 0.024 \). In higher ranges such as 0.5-0.7, it underpromises, since it predicted \( 0.598 \) but actual returns were \( 0.708 \), and in the 0.7-1.0 range, it predicted \( 0.767 \) but actual returns were \( 0.865 \).

<br>

---

<br>

**(c)** The ROC-AUC metric fails to capture this calibration problem because it only evaluates the models' ability to rank instances (i.e., whether positive records get higher scores than negative ones). The absolute value of the predictions does not affect ROC-AUC. Given this, the best decision is to calibrate the Random Forest, as it already has excellent discriminative power, and we only need to correct its probability scale to make its predictions reliable for decision-making.

<br>

---

<br>

**(d)** The cost-minimizing threshold for the Logistic Regression is \( 0.09 \) with a cost of \( 456 \), and for the Random Forest, it is \( 0.23 \) with a cost of \( 483 \). If we directly applied the logistic threshold (\( 0.09 \)) to the Forest, its total cost would increase to \( 619 \) (a major increase of \( +28.2\% \) over its optimal cost of \( 483 \)). We cannot transfer a threshold from one model to another because the distribution of estimated probabilities is fundamentally different between models; a threshold calibrated for one distribution does not carry the same probabilistic meaning in another.

##≽༏≼⊹⋅⭒.⭒⋅⊹｡ ݁.⋆₊˚⋆☾✩☽⋆˚ ⊹⋅⭒⋅⊹｡ 𓂃˖˳·˖ ִֶָ🍄⋆˳·🌸˖ ִֶָ 🌷𓍢˖·🌹˖˳·🦩˖·🎀˳⋆ ִֶָ🌺 ִֶ˖·˳˖𓂃🦋 ִֶָ݁.⋆ ≽༏≼⊹⋅⭒.⭒⋅




## Exercise 4 — The column that leaks the answer *(2.0 points)*

A bank wants to predict, **at the moment the credit card is approved**, which customers will fall behind on their credit card bills in the following 12 months (`atrasou = 1`). The dataset `q_credito.csv` came from the data lake with these columns:

| column | what it is | when it originates |
|---|---|---|
| `renda_mensal` | declared monthly income | at registration |
| `idade` | customer age | at registration |
| `score_bureau` | credit bureau score | queried during approval |
| `comprometimento` | % of income already committed to debt | at registration |
| `tempo_emprego` | years in current job | at registration |
| `dias_atraso_max` | maximum delinquency, in days, over 12 months | during the 12 months |
| `ligacoes_cobranca` | collection calls over 12 months | during the 12 months |
| `limite_reduzido` | 1 if the bank reduced the credit limit in the year | during the 12 months |
| `nota_pesquisa` | satisfaction survey rating | in the 6th month |

**(a)** Run the audit: which columns stay and which go? State the question that determines this.

**(b)** Provide the accuracy using all columns, using only the ones that stay, and using the baseline (dummy) classifier. Which number goes into the bank's report, and how much does it gain over the baseline?

**(c)** In the "one at a time" table, which columns inflate accuracy, and which column practically does not change it? Provide the values. Does the column that doesn't change also leave? Why?

**(d)** Which of the three warning signs of target leakage appear in your numbers? Cite the values.

In [7]:
df = pd.read_csv("q_credito.csv")
y = df["atrasou"]
X = df.drop(columns="atrasou")
print("rows:", len(y), "| defaulted:", f"{y.mean():.1%}")

STAY = ["renda_mensal", "idade", "score_bureau", "comprometimento", "tempo_emprego"]
AFTER = ["dias_atraso_max", "ligacoes_cobranca", "limite_reduzido", "nota_pesquisa"]

def accuracy_cv(columns):
    return cross_val_score(logistic(), X[columns], y, cv=cv, scoring="accuracy").mean()

acc_all = accuracy_cv(STAY + AFTER)
acc_stay = accuracy_cv(STAY)
acc_dummy = cross_val_score(DummyClassifier(strategy="most_frequent"), X[STAY], y,
                            cv=cv, scoring="accuracy").mean()
print(f"\nall columns     : {acc_all:.3f}")
print(f"only those that stay: {acc_stay:.3f}")
print(f"baseline (dummy): {acc_dummy:.3f}")

print("\none at a time: stay columns + current column")
print(f"{'column added':22s} {'accuracy':8s} {'difference':11s} {'correlation with target'}")
one_by_one = {}
for col in AFTER:
    acc = accuracy_cv(STAY + [col])
    corr = np.corrcoef(X[col], y)[0, 1]
    one_by_one[col] = (acc, corr)
    print(f"{col:22s} {acc:.3f}     {acc - acc_stay:+.3f}        {corr:+.3f}")

rows: 903 | defaulted: 36.1%

all columns     : 0.992
only those that stay: 0.714
baseline (dummy): 0.639

one at a time: stay columns + current column
column added           accuracy difference  correlation with target
dias_atraso_max        0.993     +0.279        +0.815
ligacoes_cobranca      0.854     +0.140        +0.635
limite_reduzido        0.766     +0.052        +0.314
nota_pesquisa          0.714     -0.000        -0.019


**Resposta**

**(a)** The variables that must be excluded from the modeling are: `dias_atraso_max`, `ligacoes_cobranca`, `limite_reduzido`, and `nota_pesquisa`. Only the columns `renda_mensal`, `idade`, `score_bureau`, `comprometimento`, and `tempo_emprego` should stay. The key question that decides the exclusion is: *"Is this information already available at the exact moment the credit card is approved?"* Information that originates during or after the 12-month period constitutes *target leakage*.

<br>

---

<br>

**(b)** The accuracy obtained using all columns was \( 0.992 \); using only the valid columns that stay, the value drops to \( 0.714 \), while the accuracy of the baseline (dummy) classifier is \( 0.639 \). The performance number that should go into the bank's final report is \( 0.714 \). The actual incremental gain of this model over the baseline is \( 0.075 \) (meaning, \( 71.4\% - 63.9\% \)).

<br>

---

<br>

**(c)** The columns that inflate performance are `dias_atraso_max` (increasing accuracy to \( 0.993 \), with a difference of \( +0.279 \) and target correlation of \( +0.815 \)) and `ligacoes_cobranca` (resulting in an accuracy of \( 0.854 \), a difference of \( +0.140 \), and target correlation of \( +0.635 \)). The column that practically does not change performance is `nota_pesquisa` (accuracy of \( 0.714 \), a difference of \( -0.000 \), and target correlation of \( -0.019 \)). Yes, `nota_pesquisa` must also be removed, because it is only filled out in the 6th month after approval, which makes it impossible to use at the time the credit card is being approved.

<br>

---

<br>

**(d)** The typical signs of target leakage present in these numbers are: an excessively perfect and unrealistic accuracy close to 100\% (\( 0.992 \) when using all columns) and the presence of features that are highly correlated with the target variable after approval has occurred, such as the correlation found in the `dias_atraso_max` column (\( +0.815 \)) and the `ligacoes_cobranca` column (\( +0.635 \)).

##≽༏≼⊹⋅⭒.⭒⋅⊹｡ ݁.⋆₊˚⋆☾✩☽⋆˚ ⊹⋅⭒⋅⊹｡ 𓂃˖˳·˖ ִֶָ🍄⋆˳·🌸˖ ִֶָ 🌷𓍢˖·🌹˖˳·🦩˖·🎀˳⋆ ִֶָ🌺 ִֶ˖·˳˖𓂃🦋 ִֶָ݁.⋆ ≽༏≼⊹⋅⭒.⭒⋅




## Exercise 5 — Four candidates, one ruler *(2.0 points)*

The dataset `q_candidatos.csv` contains customers of a service provider; the target is `alvo = 1` for those who **cancelled** their subscription. Four candidate models go through the **same 10 folds**: the baseline dummy (always predicting the most common class), a decision tree of depth 3, a random forest, and logistic regression. The output displays the ROC-AUC (mean and standard deviation across folds) and the accuracy for each.

**(a)** Which candidate has the highest ROC-AUC? Give its mean and standard deviation, and the means of the other three.

**(b)** Use the tie-break interval rule (the best mean minus one standard deviation). What is the threshold floor, which candidates fall inside, and which fall outside? What does this tell us about the difference among those inside?

**(c)** Which model do you deliver, and why? If there is a tie, state your tie-breaking criterion — and explain why it is not the third decimal place.

**(d)** Look at the accuracy of the baseline dummy classifier. Provide its value and explain why the comparison was made using ROC-AUC rather than accuracy.

In [8]:
df = pd.read_csv("q_candidatos.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("rows:", len(y), "| columns:", X.shape[1], "| cancelled:", f"{y.mean():.1%}")

candidates = {
    "dummy":     DummyClassifier(strategy="most_frequent"),
    "tree":      DecisionTreeClassifier(max_depth=3, random_state=0),
    "forest":    RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0),
    "logistic":  logistic(),
}

res = {}
print("\ncandidate    ROC-AUC   std dev   accuracy")
for name, model in candidates.items():
    r = cross_validate(model, X, y, cv=cv, scoring=["roc_auc", "accuracy"])
    res[name] = (r["test_roc_auc"].mean(), r["test_roc_auc"].std(), r["test_accuracy"].mean())
    print(f"{name:12s} {res[name][0]:.4f}   {res[name][1]:.4f}   {res[name][2]:.4f}")

best = max(res, key=lambda k: res[k][0])
floor = res[best][0] - res[best][1]
print(f"\nbest ROC-AUC: {best}  {res[best][0]:.4f} ± {res[best][1]:.4f}")
print(f"tie-break interval floor: {floor:.4f}")
print("inside the interval:", [n for n in res if res[n][0] >= floor])
print("outside the interval:", [n for n in res if res[n][0] < floor])

rows: 751 | columns: 8 | cancelled: 29.2%

candidate    ROC-AUC   std dev   accuracy
dummy        0.5000   0.0000   0.7084
tree         0.7206   0.0699   0.7524
forest       0.8529   0.0401   0.7989
logistic     0.8300   0.0493   0.7736

best ROC-AUC: forest  0.8529 ± 0.0401
tie-break interval floor: 0.8128
inside the interval: ['forest', 'logistic']
outside the interval: ['dummy', 'tree']


**Resposta:**

**(a)** The candidate with the highest average ROC-AUC is the **forest**, with a mean of \( 0.8529 \) and a standard deviation of \( 0.0401 \). The other candidates obtained the following average ROC-AUC values: the tree obtained \( 0.7206 \), the logistic regression obtained \( 0.8300 \), and the dummy baseline obtained \( 0.5000 \).

<br>

---

<br>

**(b)** Applying the tie-break interval rule, we calculate the lower limit (floor) from the best average performance (\( 0.8529 \)) minus its standard deviation (\( 0.0401 \)):

<br>

$$
0.8529 - 0.0401 = 0.8128
$$

<br>

The calculated floor value is \( 0.8128 \). The candidates inside this tie-break interval (performance greater than or equal to \( 0.8128 \)) are the **forest** (\( 0.8529 \)) and the **logistic regression** (\( 0.8300 \)). The **tree** (\( 0.7206 \)) and the **dummy** baseline (\( 0.5000 \)) fall outside the interval. This shows us that the difference in performance between the forest and the logistic regression is not statistically significant given the fold-to-fold variation.

<br>

---

<br>

**(c)** The model we should deliver is **logistic regression**. Since it is statistically tied with the forest (both are within the tie-break interval whose floor is \( 0.8128 \)), we apply the **parsimony** principle (choosing the simpler, more interpretable, and computationally lighter model). The third decimal place should not be used to decide, because variations below the standard deviation of the models (which is \( 0.0401 \) for the forest and \( 0.0493 \) for the logistic model) represent nothing more than statistical sampling noise.

<br>

----

<br>

**(d)** The accuracy obtained by the baseline dummy classifier was \( 0.708 \). This value indicates that the dataset is imbalanced (approximately \( 70.8\% \) of the instances belong to the majority class). The comparison must be made using ROC-AUC and not accuracy because accuracy is highly skewed by class imbalance, where a naive baseline model (dummy) that never predicts cancellations achieves a high score of \( 0.708 \) without actually performing any useful discrimination or ordering of customers.

```markdown
🐝
                           🐝
        ≽༏≼


          😽
         <))>
   🌱⋅⊹｡ _/  \_  🌻 ᓚᘏᗢ 🍄 ⋅∗.∗ ⁂
```